# Token Design

Goal:
Design a transformer-friendly token representation
for Riichi Mahjong policy learning.

The representation should:
- preserve game semantics,
- support dynamic legal actions,
- encode public and private information,
- and remain extensible for future sequential modeling.

# Step 1 — Information Categories

Before defining token structures, we first classify
the different types of information contained in a Mahjong state.

This helps determine:
- which entities should become tokens,
- which information should preserve order,
- and which features should be represented compositionally.

| Category          | Examples      | Ordered?  | Public? | Dynamic? |
| ----------------- | ------------- | --------- | ------- | -------- |
| Hand tiles        | 5m, east      | No        | No      | Yes      |
| Discards          | 9m discard    | Yes       | Yes     | Yes      |
| Melds             | pon/chii/kan  | Partially | Yes     | Yes      |
| Candidate actions | discard 5m    | No        | N/A     | Yes      |
| Context           | round wind    | No        | Yes     | Slow     |
| Player status     | riichi, score | No        | Yes     | Yes      |


# Step 2 — Ordered vs Unordered Information

Different parts of Mahjong state contain different
structural properties.

Some information is fundamentally sequential and should
preserve temporal ordering, while other information is
set-like and order-invariant.


# Step 3 — Token Types

The Mahjong state will be represented as a collection
of semantic tokens/entities.

Each token type represents a different category of
game information and may contain different metadata fields.

## Hand Token Representation

Hand tiles are represented as semantic tile-instance tokens.

The representation preserves:
- tile identity,
- duplicate distinction,
- and future compatibility with red-five handling.

Rather than using the raw dataset tile IDs directly,
tiles are converted into canonical semantic instances.

For example:

- 1m_copy0
- 1m_copy1
- east_copy2

This avoids learning arbitrary dataset-specific indices
while still preserving physical tile multiplicity.

Each hand token is represented compositionally through:
- token type,
- tile type,
- copy index,
- and future optional metadata such as red-five flags.

| Field      | Purpose                  |
| ---------- | ------------------------ |
| token_type | identifies HAND token    |
| tile_type  | semantic Mahjong tile    |
| copy_index | distinguishes duplicates |
| is_red     | red dora support         |


## Discard Tokens

Discard tokens represent ordered public discard events.

Unlike concealed hand tiles, discard events are fundamentally sequential and preserve temporal information.

Discard tokens contain both:
- semantic tile information,
- and contextual metadata about how and when the discard occurred.

| Field                 | Purpose                                    |
| --------------------- | ------------------------------------------ |
| token_type            | identifies DISCARD token                   |
| player                | player who discarded                       |
| tile_type             | semantic tile identity                     |
| copy_index            | duplicate distinction                      |
| is_red                | red dora support                           |
| is_tsumogiri          | whether discard was immediately drawn tile |
| is_riichi_declaration | whether discard declared riichi            |
| global_order          | chronological game order                   |
| player_discard_order  | discard index for that player              |


## Meld Tokens

Meld tokens represent open calls made during gameplay.

Unlike hand tiles, melds are public structured actions
that contain both:
- tile composition,
- and relational information about the call itself.

Melds are represented as semantic event entities rather
than independent tile tokens.

| Field          | Purpose                   |
| -------------- | ------------------------- |
| token_type     | identifies MELD token     |
| player         | player who owns meld      |
| meld_type      | chi / pon / kan           |
| tile_types     | semantic tile composition |
| copy_indices   | duplicate distinction     |
| red_flags      | red dora support          |
| source_players | who contributed each tile |
| global_order   | chronological call timing |


## Action Tokens

Action tokens represent candidate legal decisions available to the acting player.

Unlike static tile representations, action tokens are dynamic entities whose meaning depends heavily on game context.

The policy-learning problem is formulated as ranking legal candidate actions rather than predicting tiles directly.

| Field          | Purpose                          |
| -------------- | -------------------------------- |
| token_type     | identifies ACTION token          |
| acting_player  | player taking action             |
| action_index   | index inside legal candidate set |
| action_type    | discard / riichi / pon / etc     |
| tile_types     | tiles involved in action         |
| copy_indices   | duplicate distinction            |
| red_flags      | red dora support                 |
| source_players | source of called tiles           |
| global_order   | chronological timing             |


## Game State Tokens

Game-state tokens represent persistent global context information shared by all players.

These tokens provide environmental context that influences strategic decisions throughout the game.

| Field           | Purpose                     |
| --------------- | --------------------------- |
| token_type      | identifies GAME_STATE token |
| round_wind      | East/South round            |
| honba_count     | current honba               |
| riichi_sticks   | riichi deposits             |
| remaining_tiles | tiles left in wall          |
| dora_indicators | current dora indicators     |


## Player State Tokens

Player-state tokens represent persistent contextual information associated with each player.

Unlike discard or meld events, these tokens encode stable strategic state such as:
- score,
- riichi status,
- seat position,
- and placement pressure.

| Field         | Purpose                        |
| ------------- | ------------------------------ |
| token_type    | identifies PLAYER_STATE token  |
| player        | player identity                |
| seat_wind     | east/south/west/north          |
| score         | current points                 |
| riichi_status | whether player declared riichi |
| placement     | current ranking                |
| open_hand     | whether player has opened hand |


# Step 4 — Embedding Composition Design

Tokens are represented compositionally rather than as monolithic symbolic IDs.

Each token embedding is constructed from multiple semantic embedding components.

This design:
- improves generalization,
- preserves semantic structure,
- reduces vocabulary explosion,
- and supports extensibility for future architectures.

## General Embedding Structure

Each token embedding is composed as:

Embedding(token) =
    token_type_embedding
    + semantic_embeddings
    + metadata_embeddings
    + positional_embeddings

Different token types use different subsets of embeddings depending on their semantic structure.

## Token Type Embedding

Each token category receives a dedicated token-type embedding.

Examples:
- HAND
- DISCARD
- MELD
- ACTION
- PLAYER_STATE
- GAME_STATE

This allows the transformer to distinguish semantic roles between tokens.

## Tile Embedding

Tile embeddings encode the semantic identity of Mahjong tiles.

Examples:
- 1m
- 5p
- east
- red dragon

Tile embeddings are shared across multiple token categories:
- hand tokens,
- discard tokens,
- meld tokens,
- and action tokens.

## Copy Index Embedding

Duplicate tile instances are distinguished through copy-index embeddings.

Examples:
- first 1m
- second 1m
- third 1m

This preserves:
- duplicate distinction,
- legal-action consistency,
- and future replay reconstruction compatibility.

## Player Embedding

Player embeddings encode player identity.

These embeddings are used in:
- discard events,
- meld events,
- action tokens,
- and player-state tokens.

This allows the model to reason about:
- opponent behavior,
- relative danger,
- and multi-agent interactions.

## Positional Embeddings

Sequential event tokens preserve temporal order through positional embeddings.

This is especially important for:
- discard sequences,
- meld timing,
- and future trajectory extensions.

Different positional schemes may coexist:
- global chronological order,
- and player-local discard order.

## Metadata Embeddings

Additional strategic metadata is represented through dedicated embeddings.

Examples:
- tsumogiri flag,
- riichi declaration discard,
- meld type,
- red dora indicator,
- action type.

## Hand Token Embedding

Hand tokens represent concealed tiles currently held by the acting player.

The embedding of a hand token is defined as:

E_hand =
    E_token_type
    + E_tile
    + E_copy
    + E_red

where:

- E_token_type identifies the token as a HAND entity,
- E_tile represents semantic tile identity,
- E_copy distinguishes duplicate tile instances,
- E_red indicates whether the tile is a red dora.

## Discard Token Embedding

Discard tokens represent ordered public discard events.

The embedding of a discard token is defined as:

E_discard =
    E_token_type
    + E_tile
    + E_copy
    + E_red
    + E_player
    + E_tsumogiri
    + E_riichi_declaration
    + E_global_position
    + E_local_position

where:

- E_token_type identifies the token as a DISCARD entity,
- E_tile represents semantic tile identity,
- E_copy distinguishes duplicate tile instances,
- E_red indicates red dora status,
- E_player identifies the discarding player,
- E_tsumogiri indicates whether the discarded tile was immediately drawn,
- E_riichi_declaration indicates whether the discard declared riichi,
- E_global_position represents overall chronological order,
- E_local_position represents discard order relative to the player.

## Meld Token Embedding

Meld tokens represent open calls made during gameplay.

The embedding of a meld token is defined as:

E_meld =
    E_token_type
    + E_player
    + E_meld_type
    + E_tile_set
    + E_copy_set
    + E_red_set
    + E_source_players
    + E_global_position

where:

- E_token_type identifies the token as a MELD entity,
- E_player identifies the meld owner,
- E_meld_type represents chi, pon, kan, etc,
- E_tile_set encodes the tiles composing the meld,
- E_copy_set distinguishes duplicate instances,
- E_red_set encodes red dora information,
- E_source_players identifies tile origins,
- E_global_position represents the timing of the call.

## Action Token Embedding

Action tokens represent candidate legal actions available to the acting player.

The embedding of an action token is defined as:

E_action =
    E_token_type
    + E_action_type
    + E_tile_set
    + E_copy_set
    + E_red_set
    + E_source_players
    + E_global_position

where:

- E_token_type identifies the token as an ACTION entity,
- E_action_type represents discard, riichi, chi, pon, kan, etc,
- E_tile_set represents the tiles involved in the action,
- E_copy_set distinguishes duplicate tile instances,
- E_red_set encodes red dora information,
- E_source_players identifies tile origins for calls,
- E_global_position represents action timing.

## Game State Token Embedding

Game-state tokens represent persistent global context information.

The embedding of a game-state token is defined as:

E_game =
    E_token_type
    + E_round
    + E_honba
    + E_riichi_sticks
    + E_remaining_tiles
    + E_dora_set

## Player State Token Embedding

Player-state tokens represent persistent contextual information associated with each player.

The embedding of a player-state token is defined as:

E_player_state =
    E_token_type
    + E_player
    + E_seat
    + E_score
    + E_riichi_status
    + E_open_hand

# Step 5 — Tokenization Strategy

The Mahjong state is converted into a unified sequence of semantic tokens.

Each decision point corresponds to one transformer input sequence containing:
- global game context,
- player context,
- private hand information,
- public events,
- and candidate legal actions.

## Candidate Action Scoring

Candidate action tokens are appended at the end of the token sequence.

The transformer produces contextualized embeddings for each action token.

Policy logits are computed independently from the final hidden representations of candidate action tokens.

A softmax over candidate-action logits produces the policy distribution over legal actions.

# Step 6 — Tensorization Mechanics

Semantic tokens are converted into structured tensor fields rather than monolithic token IDs.

Each semantic attribute receives an independent embedding table.

This enables:
- compositional representation learning,
- parameter sharing,
- and flexible semantic generalization.

## Tensor Fields

Each token is represented through multiple tensorized feature fields.

Examples include:
- token type IDs,
- tile IDs,
- player IDs,
- positional indices,
- metadata flags,
- and action types.

| Field          | Tensor Type | Example     |
| -------------- | ----------- | ----------- |
| token_type_id  | int         | HAND        |
| tile_id        | int         | 5m          |
| copy_id        | int         | second copy |
| player_id      | int         | P2          |
| tsumogiri_id   | binary int  | True        |
| action_type_id | int         | riichi      |
| position_id    | int         | discard #12 |


## Token Sequence Tensorization

Each Mahjong decision point is represented as a sequence of tokens.

The transformer input consists of:
- a token dimension,
- and multiple parallel feature fields per token.

## Padding Strategy

Mahjong states contain variable numbers of:
- discard events,
- melds,
- and candidate actions.

Sequences are padded to a fixed maximum length during batching.

Padding tokens receive:
- dedicated PAD token types,
- and are masked during transformer attention.

## Candidate Action Alignment

Candidate action tokens are appended at the end of the token sequence.

Their positions are tracked explicitly during tensorization.

The transformer outputs contextualized embeddings for all tokens, but policy logits are computed only from candidate-action token positions.